# CRISP-DM Fase 3: Data Preparation - Limpieza, Wrangling y Gobernanza PII
## Dataset: SIPSA Abastecimiento de Alimentos (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 3: Data Cleaning & Transformation) | **Fase PDCO**: DEVELOPMENT  
**Estándares**: DAMA-DMBOK 2 (Data Security & Governance), Ley 1581 de 2012 (Habeas Data Colombia).

---

### 🧹 1. Pipeline de Estandarización y Limpieza
* Normalización de identificadores a formato estándar `snake_case`.
* Tratamiento de cadenas de texto (eliminación de espacios en blanco y caracteres de control).
* Imputación estadística de valores faltantes.

### 🔒 2. Cumplimiento de Privacidad y PII (Personally Identifiable Information)
* Detección automática y anonimización irreversible mediante hashing **SHA-256** con salt para nombres, correos, documentos o datos de contacto de productores agropecuarios.
* Persistencia en formato columnar **Parquet con compresión Snappy** (Capa Silver).

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 5: NORMALIZACIÓN SNAKE_CASE Y SANITIZACIÓN DE CADENAS]
# ==============================================================================
from src.cleaning.sanitizer import DataSanitizer
from src.cleaning.pii_handler import PIIHandler

parquet_file = APP_ROOT / "data/processed/sipsa_abastecimientos.parquet"
df = pd.read_parquet(parquet_file)

print(f"📦 Columnas antes de limpieza: {list(df.columns)[:5]}...")

# 1. Limpieza de nombres de columna
df_clean = DataSanitizer.clean_column_names(df)

# 2. Tratamiento de espacios y strings
df_clean = DataSanitizer.sanitize_strings(df_clean)

# 3. Anonimización de PII si existen datos personales (Ley 1581)
pii_columns = [col for col in df_clean.columns if any(p in col.lower() for p in ['email', 'correo', 'telefono', 'nombre', 'contacto', 'identificacion'])]
if pii_columns:
    print(f"🔒 Columnas PII sensibles detectadas: {pii_columns} -> Aplicando Hashing SHA-256")
    for pii_col in pii_columns:
        df_clean[pii_col] = PIIHandler.anonymize_series(df_clean[pii_col])
else:
    print("ℹ️ No se detectaron campos de datos personales sensibles (PII). Cumple Ley 1581.")

print(f"✅ DataFrame sanitizado. Dimensiones resultantes: {df_clean.shape}")


In [ ]:
# ==============================================================================
# [FASE 5: EXPORTACIÓN A CAPA SILVER EN PARQUET COMPRIMIDO SNAPPY]
# ==============================================================================
out_parquet = APP_ROOT / "data/processed/sipsa_abastecimientos.parquet"
out_parquet.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(out_parquet, engine="pyarrow", compression="snappy", index=False)
file_size_kb = out_parquet.stat().st_size / 1024

print(f"💾 Archivo persistido en Capa Silver:")
print(f"• Ruta: {out_parquet.relative_to(APP_ROOT)}")
print(f"• Tamaño en disco: {file_size_kb:.2f} KB")
print(f"• Registros válidos: {len(df_clean):,} filas")
